# Week 3 · Course 5 · Moving Beyond Linearity

Linear models are powerful — but the real world rarely is linear.  
This notebook covers five families of techniques that extend the linear model with **flexible nonlinear fits** while remaining interpretable:

1. **Polynomial regression** — higher-degree terms
2. **Step functions** — piecewise constant fits
3. **Splines** — smooth piecewise polynomials (linear, cubic, natural cubic)
4. **Smoothing splines** — roughness-penalised fits
5. **Local regression (LOESS)** — sliding weighted windows
6. **Generalized Additive Models (GAMs)** — additive nonlinear components

All methods fit inside standard `LinearRegression` / `LogisticRegression` — you transform predictors, then run OLS as usual.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import UnivariateSpline
from scipy.special import expit
from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut
from sklearn.metrics import mean_squared_error

sns.set_theme(style='whitegrid', palette='muted')
PALETTE = sns.color_palette('muted')
C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE = PALETTE[:5]
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

print('All libraries loaded ✓')

## Synthetic Wage dataset

Throughout this notebook we use a synthetic Wage dataset (n=3000) inspired by the ISLR `Wage` data:  
- **age** (18–80): primary predictor, nonlinear relationship with wage  
- **year** (2003–2009): slight linear trend  
- **education** (5 levels): strong stepwise effect  
- **wage**: roughly ∝ a degree-2 polynomial in age + education bump + noise

In [ ]:
def make_wage(n=3000, seed=42):
    rng = np.random.default_rng(seed)
    age  = rng.integers(18, 81, size=n).astype(float)
    year = rng.integers(2003, 2010, size=n).astype(float)
    edu_levels = ['<HS', 'HS', '<Coll', 'Coll', '>Coll']
    edu_idx    = rng.choice(5, size=n, p=[0.10, 0.25, 0.20, 0.25, 0.20])
    education  = np.array(edu_levels)[edu_idx]
    edu_effect = np.array([-15, 0, 10, 20, 35])[edu_idx]
    age_effect = -0.015*(age-49)**2 + 0.3*(age-49)
    year_effect= 1.5*(year-2006)
    wage = 110 + age_effect + year_effect + edu_effect + rng.normal(0, 20, n)
    wage = np.clip(wage, 20, 320)
    high = rng.random(n) < 0.02
    wage[high] = rng.uniform(250, 320, high.sum())
    return pd.DataFrame({'age': age, 'year': year, 'wage': wage, 'education': education})

df = make_wage()
print(df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(df['age'],  df['wage'], alpha=0.15, s=15)
axes[0].set(xlabel='Age', ylabel='Wage', title='Wage vs Age')
axes[1].scatter(df['year'], df['wage'], alpha=0.15, s=15)
axes[1].set(xlabel='Year', ylabel='Wage', title='Wage vs Year')
edu_order = ['<HS', 'HS', '<Coll', 'Coll', '>Coll']
df_plot = df.copy()
df_plot['education'] = pd.Categorical(df_plot['education'], categories=edu_order, ordered=True)
df_plot.sort_values('education').boxplot('wage', by='education', ax=axes[2])
axes[2].set(xlabel='Education', ylabel='Wage', title='Wage vs Education')
plt.suptitle('')
plt.tight_layout()
plt.show()

---
## Part 1 — Polynomial Regression

Polynomial regression extends the linear model by including higher powers of X as features:

$$y_i = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \cdots + \beta_d x_i^d + \varepsilon_i$$

This is still a **linear model** — linear in the coefficients β.  
We just create new columns X², X³, … and run standard OLS.

In [ ]:
# --- 1a. Fit degrees 1-4 with sklearn Pipeline ---
age = df['age'].values
wage = df['wage'].values
x_rng = np.linspace(age.min(), age.max(), 300)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = [C_BLUE, C_ORANGE, C_GREEN, C_RED]

for deg, ax, color in zip([1, 2, 3, 4], axes.ravel(), colors):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    pipe.fit(age.reshape(-1,1), wage)
    y_hat = pipe.predict(x_rng.reshape(-1,1))
    train_r2 = pipe.score(age.reshape(-1,1), wage)
    
    ax.scatter(age, wage, alpha=0.15, s=15, color='steelblue')
    ax.plot(x_rng, y_hat, color=color, lw=2.5, label=f'Degree {deg}')
    ax.set(xlabel='Age', ylabel='Wage', title=f'Degree-{deg} Polynomial  (R²={train_r2:.3f})')
    ax.legend()

plt.suptitle('Polynomial Regression: Degrees 1–4', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Confidence Intervals with statsmodels

`statsmodels.OLS` gives us standard errors and confidence intervals for the fitted curve.

In [ ]:
# Fit degree-4 polynomial with statsmodels for proper CIs
poly4 = PolynomialFeatures(degree=4, include_bias=True)
X4 = poly4.fit_transform(age.reshape(-1,1))
model_sm = sm.OLS(wage, X4).fit()
print(model_sm.summary().tables[1])

In [ ]:
X4_rng = poly4.transform(x_rng.reshape(-1,1))
pred    = model_sm.get_prediction(X4_rng)
pred_df = pred.summary_frame(alpha=0.05)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(age, wage, alpha=0.18, s=18, color='steelblue')
ax.plot(x_rng, pred_df['mean'], color=C_ORANGE, lw=2.5, label='Degree-4 fit')
ax.fill_between(x_rng, pred_df['mean_ci_lower'], pred_df['mean_ci_upper'],
                alpha=0.25, color=C_ORANGE, label='95% CI')
ax.set(xlabel='Age', ylabel='Wage', title='Degree-4 Polynomial with 95% Confidence Band')
ax.legend()
plt.tight_layout()
plt.show()

### Logistic Polynomial — Pr(Wage > 250 | Age)

In [ ]:
y_bin = (wage > 250).astype(int)
print(f'High earners: {y_bin.sum()} / {len(y_bin)} ({y_bin.mean()*100:.1f}%)')

pipe_logit = Pipeline([
    ('poly', PolynomialFeatures(degree=4, include_bias=False)),
    ('lr',   LogisticRegression(C=1e4, max_iter=1000))
])
pipe_logit.fit(age.reshape(-1,1), y_bin)
p_hat = pipe_logit.predict_proba(x_rng.reshape(-1,1))[:, 1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_rng, p_hat, color=C_BLUE, lw=2.5, label='Pr(Wage > 250 | Age)')
ax.plot(age[y_bin==1], np.full(y_bin.sum(), 0.001), '|', color=C_ORANGE, ms=10, alpha=0.7)
ax.set(xlabel='Age', ylabel='Probability', ylim=(-0.01, 0.22),
       title='Logistic Degree-4 Polynomial: Pr(Wage > 250 | Age)')
ax.legend()
plt.tight_layout()
plt.show()

### Choosing the degree by cross-validation

In [ ]:
degrees = range(1, 11)
cv_mse = []
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    scores = cross_val_score(pipe, age.reshape(-1,1), wage,
                             cv=kf, scoring='neg_mean_squared_error')
    cv_mse.append(-scores.mean())

best_d = degrees[np.argmin(cv_mse)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(degrees), cv_mse, 'o-', color=C_BLUE, lw=2)
ax.axvline(best_d, color=C_RED, lw=1.5, linestyle='--', label=f'Best degree = {best_d}')
ax.set(xlabel='Polynomial Degree', ylabel='10-fold CV MSE',
       title='Cross-Validation to Choose Polynomial Degree')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Best degree by CV: {best_d}')

---
## Part 2 — Step Functions

Cut the range of X into K+1 discrete bins using fixed cut points c₁ < c₂ < … < cₖ.  
Create indicator dummy variables for each bin — then run OLS.  

Each region gets its own intercept (constant level).  
**Key use**: creating interpretable interaction terms, e.g. different slopes before/after a policy change.

In [ ]:
cuts   = [18, 33.5, 49, 64.5, 80]
labels = ['18–33', '33–49', '49–65', '65–80']

age_cut = pd.cut(age, bins=cuts, labels=labels, include_lowest=True)
dummies = pd.get_dummies(age_cut, drop_first=False).astype(float)
print(dummies.head())

m_step = LinearRegression(fit_intercept=True).fit(dummies.values, wage)
print('\nFitted means per bin:')
for lab, coef in zip(labels, m_step.coef_):
    print(f'  {lab}: {m_step.intercept_ + coef:.1f}')

In [ ]:
x_rng = np.linspace(18, 80, 600)
cat_rng = pd.cut(x_rng, bins=cuts, labels=labels, include_lowest=True)
d_rng   = pd.get_dummies(cat_rng, drop_first=False).astype(float)
y_step  = m_step.predict(d_rng.values)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(age, wage, alpha=0.18, s=18, color='steelblue')
ax.step(x_rng, y_step, where='mid', color=C_GREEN, lw=2.5, label='Step function')
for c in cuts[1:-1]:
    ax.axvline(c, color='grey', lw=0.8, linestyle='--', alpha=0.6)
ax.set(xlabel='Age', ylabel='Wage', title='Step Function: Piecewise Constant Fit')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 3 — Splines

Splines combine the best of step functions and polynomials:  
- **Piecewise** polynomial fit (one polynomial per region)
- **Continuous** at the knots — no jumps or kinks

Key concept: the **truncated power basis** adds one column per knot:

$$b_{k+1}(x) = (x - \xi_k)_+ = \max(x - \xi_k, 0)$$

For cubic splines, raise to the power 3.

In [ ]:
def truncated_power_basis(x, knots, degree=3):
    """Build truncated power basis for spline fitting."""
    cols = [x**d for d in range(1, degree+1)]
    for k in knots:
        cols.append(np.maximum(x - k, 0)**degree)
    return np.column_stack(cols)

# Demonstrate the truncated cubic basis
xi = 49.0
x_demo = np.linspace(18, 80, 300)
b_trunc = np.maximum(x_demo - xi, 0)**3

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_demo[x_demo < xi], b_trunc[x_demo < xi], color=C_BLUE, lw=2.5)
axes[0].plot(x_demo[x_demo >= xi], b_trunc[x_demo >= xi], color=C_ORANGE, lw=2.5)
axes[0].axvline(xi, color='grey', lw=0.9, linestyle=':', label=f'Knot ξ={xi}')
axes[0].set(xlabel='x', ylabel='b(x)', title='Truncated Cubic Basis Function')
axes[0].legend()

# Linear truncated basis for comparison
b_lin = np.maximum(x_demo - xi, 0)
axes[1].plot(x_demo[x_demo < xi], b_lin[x_demo < xi], color=C_BLUE, lw=2.5)
axes[1].plot(x_demo[x_demo >= xi], b_lin[x_demo >= xi], color=C_ORANGE, lw=2.5)
axes[1].axvline(xi, color='grey', lw=0.9, linestyle=':', label=f'Knot ξ={xi}')
axes[1].set(xlabel='x', ylabel='b(x)', title='Truncated Linear Basis Function')
axes[1].legend()

plt.suptitle('Truncated Power Basis Functions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Linear Spline

In [ ]:
x_rng = np.linspace(age.min(), age.max(), 300)
knot_single = np.array([49.0])

X_lin = truncated_power_basis(age, knot_single, degree=1)
m_lin = LinearRegression(fit_intercept=True).fit(X_lin, wage)
y_lin = m_lin.predict(truncated_power_basis(x_rng, knot_single, degree=1))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(age, wage, alpha=0.18, s=18, color='steelblue')
ax.plot(x_rng, y_lin, color=C_PURPLE, lw=2.5, label='Linear Spline (1 knot at 49)')
ax.axvline(49, color='grey', lw=0.8, linestyle='--', alpha=0.7)
ax.set(xlabel='Age', ylabel='Wage', title='Linear Spline')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Coefficients: intercept={m_lin.intercept_:.2f}, slope={m_lin.coef_[0]:.3f}, kink={m_lin.coef_[1]:.3f}')

### Cubic Spline with multiple knots

In [ ]:
# Place knots at quartiles
knots_q = np.quantile(age, [0.25, 0.50, 0.75])
print(f'Knots at quantiles 25/50/75: {knots_q}')

X_cub = truncated_power_basis(age, knots_q, degree=3)
m_cub = LinearRegression(fit_intercept=True).fit(X_cub, wage)
y_cub = m_cub.predict(truncated_power_basis(x_rng, knots_q, degree=3))

print(f'Cubic spline df = K + 4 = {len(knots_q)} + 4 = {len(knots_q)+4}')
print(f'Number of basis columns: {X_cub.shape[1]} (+ intercept = {X_cub.shape[1]+1} params)')

In [ ]:
# Compare cubic splines with different numbers of knots
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = [C_BLUE, C_ORANGE, C_RED]
quantile_sets = [[0.5], [0.25, 0.5, 0.75], [0.1, 0.2, 0.4, 0.6, 0.8, 0.9]]

for ax, q_set, color in zip(axes, quantile_sets, colors):
    knots = np.quantile(age, q_set)
    X = truncated_power_basis(age, knots)
    m = LinearRegression(fit_intercept=True).fit(X, wage)
    y_hat = m.predict(truncated_power_basis(x_rng, knots))
    
    ax.scatter(age, wage, alpha=0.15, s=15, color='steelblue')
    ax.plot(x_rng, y_hat, color=color, lw=2.5,
            label=f'{len(knots)} knots ({len(knots)+4} df)')
    for k in knots:
        ax.axvline(k, color='grey', lw=0.7, linestyle=':', alpha=0.6)
    ax.set(xlabel='Age', ylabel='Wage', title=f'{len(knots)} knot(s)')
    ax.legend(fontsize=9)

plt.suptitle('Cubic Splines with Different Numbers of Knots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Natural Cubic Spline

A **natural cubic spline** adds the constraint that f(x) is linear beyond the boundary knots.  
This adds 4 constraints, reducing df by 4 — with the same df we can place more internal knots.

We use `scipy.interpolate.UnivariateSpline` with boundary conditions enforced.

In [ ]:
age_s  = np.sort(age)
wage_s = wage[np.argsort(age)]

# Natural cubic spline via scipy (smoothing controls effective df)
# s = n * sigma^2 rule-of-thumb; smaller s → more knots → more flexible
spl_nat = UnivariateSpline(age_s, wage_s, k=3, s=len(age)*100)

# Regular cubic spline (same number of knots for comparison)
knots_q4 = np.quantile(age, [0.2, 0.4, 0.6, 0.8])
X_q4 = truncated_power_basis(age, knots_q4)
m_q4 = LinearRegression(fit_intercept=True).fit(X_q4, wage)
y_q4 = m_q4.predict(truncated_power_basis(x_rng, knots_q4))

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(age, wage, alpha=0.18, s=18, color='steelblue')
ax.plot(x_rng, spl_nat(x_rng), color=C_RED, lw=2.5, label='Natural Cubic Spline')
ax.plot(x_rng, y_q4, color=C_BLUE, lw=2.0, linestyle='--', label='Cubic Spline (4 knots)')
for k in knots_q4:
    ax.axvline(k, color='grey', lw=0.7, linestyle=':', alpha=0.6)
ax.set(xlabel='Age', ylabel='Wage', title='Natural Cubic Spline vs Cubic Spline')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 4 — Smoothing Splines

Smoothing splines solve the penalised regression problem:

$$\min_{g} \sum_{i=1}^{n}(y_i - g(x_i))^2 + \lambda \int g''(t)^2 \, dt$$

- **Left term**: RSS — fits the data
- **Right term**: roughness penalty — penalises curvature
- **λ controls** the bias-variance trade-off: λ=0 → interpolation, λ→∞ → linear

The solution is a **natural cubic spline** with knots at every unique xᵢ, with λ controlling smoothness via the effective degrees of freedom.

In [ ]:
# UnivariateSpline: s controls smoothing (analogous to 1/λ)
# s=0 → interpolation; s large → very smooth
smoothing_levels = [
    (len(age) * 10,   'Very wiggly (high df)',  C_RED),
    (len(age) * 150,  'Moderate (mid df)',       C_ORANGE),
    (len(age) * 2000, 'Very smooth (low df)',    C_BLUE),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (s_val, label, color) in zip(axes, smoothing_levels):
    spl = UnivariateSpline(age_s, wage_s, k=3, s=s_val)
    n_knots = len(spl.get_knots())
    ax.scatter(age, wage, alpha=0.18, s=15, color='grey')
    ax.plot(x_rng, spl(x_rng), color=color, lw=2.5, label=label)
    ax.set(xlabel='Age', ylabel='Wage', title=f'{label}\n({n_knots} knots)')
    ax.legend(fontsize=9)

plt.suptitle('Smoothing Splines — Effect of Smoothing Parameter (s ∝ 1/λ)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Choosing λ by Leave-One-Out Cross-Validation

In [ ]:
# Grid search over smoothing parameter s
s_grid = np.logspace(2, 6, 40)
loo_mse = []

for s_val in s_grid:
    spl = UnivariateSpline(age_s, wage_s, k=3, s=s_val)
    y_hat_all = spl(age_s)
    residuals = wage_s - y_hat_all
    # Approximate LOOCV via GCV (generalised cross-validation)
    n = len(age_s)
    n_knots = max(len(spl.get_knots()), 1)
    df_eff = n_knots + 4  # rough approximation
    gcv = np.mean(residuals**2) / (1 - df_eff/n)**2
    loo_mse.append(gcv)

best_s = s_grid[np.argmin(loo_mse)]
spl_best = UnivariateSpline(age_s, wage_s, k=3, s=best_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].semilogx(s_grid, loo_mse, 'o-', color=C_BLUE, markersize=4)
axes[0].axvline(best_s, color=C_RED, lw=1.5, linestyle='--', label=f'Best s={best_s:.0f}')
axes[0].set(xlabel='Smoothing parameter s', ylabel='GCV Score',
            title='GCV to Choose Smoothing Parameter')
axes[0].legend()

axes[1].scatter(age, wage, alpha=0.18, s=18, color='grey')
axes[1].plot(x_rng, spl_best(x_rng), color=C_RED, lw=2.5,
             label=f'Best smoothing spline (s={best_s:.0f})')
axes[1].set(xlabel='Age', ylabel='Wage', title='GCV-Optimal Smoothing Spline')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Optimal s = {best_s:.1f}, knots = {len(spl_best.get_knots())}')

---
## Part 5 — Local Regression (LOESS)

**Algorithm** at each query point x₀:
1. Find the s·n nearest neighbours
2. Assign weights via the tricube kernel: $w_i = \left(1 - \left|\frac{x_i - x_0}{d_{\max}}\right|^3\right)^3$
3. Fit a weighted least-squares regression on the local window
4. ŷ(x₀) = the fitted value from the local regression

**Span** controls smoothness: small span → many local fits → flexible; large span → global fit → smooth.

In [ ]:
# statsmodels.lowess: very fast, handles large datasets
spans = [0.1, 0.3, 0.5, 0.7]
colors_lo = [C_RED, C_ORANGE, C_BLUE, C_GREEN]

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(age, wage, alpha=0.18, s=18, color='steelblue', label='Data')

for frac, color in zip(spans, colors_lo):
    smooth = lowess(wage, age, frac=frac, return_sorted=True)
    ax.plot(smooth[:, 0], smooth[:, 1], lw=2.5, color=color,
            label=f'span = {frac}')

ax.set(xlabel='Age', ylabel='Wage',
       title='Local Regression (LOESS) — Different Span Values')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Manual LOESS implementation to show the mechanics
def manual_loess(x_query, x, y, frac=0.3):
    """Manually compute LOESS at x_query points."""
    n = len(x)
    h = int(np.ceil(frac * n))
    y_hat = np.zeros(len(x_query))
    
    for i, xq in enumerate(x_query):
        dists   = np.abs(x - xq)
        idx     = np.argsort(dists)[:h]
        d_max   = dists[idx[-1]]
        weights = (1 - (dists[idx] / d_max)**3)**3  # tricube
        
        X_loc = np.column_stack([np.ones(h), x[idx]])
        W     = np.diag(weights)
        XtW   = X_loc.T @ W
        beta  = np.linalg.solve(XtW @ X_loc + 1e-8*np.eye(2), XtW @ y[idx])
        y_hat[i] = beta[0] + beta[1] * xq
    
    return y_hat

# Show the local window at a query point
xq = 45.0
frac = 0.3
n = len(age)
h = int(np.ceil(frac * n))
dists   = np.abs(age - xq)
idx     = np.argsort(dists)[:h]
d_max   = dists[idx[-1]]
weights = (1 - (dists[idx] / d_max)**3)**3

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(age, wage, alpha=0.2, s=15, color='steelblue')
ax.axvspan(age[idx].min(), age[idx].max(), alpha=0.12, color=C_ORANGE)
ax.scatter(age[idx], wage[idx], s=weights*80, color=C_ORANGE, alpha=0.7,
           edgecolors='white', linewidths=0.3, label='Local window (size ∝ weight)')
ax.axvline(xq, color=C_RED, lw=1.5, linestyle='--', label=f'Query point x₀={xq}')

# Local linear fit
X_loc = np.column_stack([np.ones(h), age[idx]])
W = np.diag(weights)
beta = np.linalg.solve(X_loc.T@W@X_loc + 1e-8*np.eye(2), X_loc.T@W@wage[idx])
x_win = np.linspace(age[idx].min(), age[idx].max(), 50)
ax.plot(x_win, beta[0] + beta[1]*x_win, color=C_ORANGE, lw=2.0, label='Local linear fit')

# Global LOESS
x_q_grid = np.linspace(age.min(), age.max(), 150)
y_lo = manual_loess(x_q_grid, age, wage, frac=frac)
ax.plot(x_q_grid, y_lo, color=C_BLUE, lw=2.0, label=f'LOESS (span={frac})')

ax.set(xlabel='Age', ylabel='Wage', title=f'Local Regression Mechanics at x₀={xq}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 6 — Generalized Additive Models (GAMs)

GAMs extend linear models by allowing each predictor to enter through a nonlinear **smooth function**:

$$y_i = \beta_0 + f_1(x_{i1}) + f_2(x_{i2}) + \cdots + f_p(x_{ip}) + \varepsilon_i$$

The **additive structure** means each component is interpretable independently — plot f_j(x_j) vs x_j.

We implement via **backfitting** or equivalently by building a single feature matrix with spline columns for each predictor.

In [ ]:
# Build GAM feature matrix: spline on age, linear year, dummies for education
age_arr  = df['age'].values
year_arr = df['year'].values
edu_arr  = df['education'].values
wage_arr = df['wage'].values

# f1(year): linear (year is nearly linear in the data)
X_year = (year_arr - year_arr.mean()).reshape(-1, 1)

# f2(age): cubic spline with 3 knots
knots_age = np.quantile(age_arr, [0.25, 0.50, 0.75])
X_age = truncated_power_basis(age_arr, knots_age, degree=3)

# f3(education): dummy variables (drop first to avoid multicollinearity)
edu_dummies = pd.get_dummies(edu_arr, drop_first=True).values.astype(float)

X_gam = np.column_stack([X_year, X_age, edu_dummies])
m_gam = LinearRegression(fit_intercept=True).fit(X_gam, wage_arr)
print(f'GAM R² = {m_gam.score(X_gam, wage_arr):.4f}')
print(f'Feature matrix shape: {X_gam.shape}')

In [ ]:
# Extract and plot each component
n_year = 1
n_age  = X_age.shape[1]

year_rng = np.linspace(year_arr.min(), year_arr.max(), 100)
f1 = m_gam.coef_[0] * (year_rng - year_arr.mean())

age_rng = np.linspace(age_arr.min(), age_arr.max(), 200)
X_age_rng = truncated_power_basis(age_rng, knots_age)
f2 = X_age_rng @ m_gam.coef_[n_year : n_year + n_age]
f2 -= f2.mean()  # centre for interpretability

edu_levels = ['<HS', 'HS', '<Coll', 'Coll', '>Coll']
edu_coefs  = np.concatenate([[0.0], m_gam.coef_[n_year + n_age:]])
edu_coefs -= edu_coefs.mean()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].plot(year_rng, f1, color=C_RED, lw=2.5)
axes[0].axhline(0, color='grey', lw=0.7, linestyle='--')
axes[0].set(xlabel='Year', ylabel='f₁(year)', title='GAM Component: Year (linear)')

axes[1].plot(age_rng, f2, color=C_RED, lw=2.5)
axes[1].axhline(0, color='grey', lw=0.7, linestyle='--')
for k in knots_age:
    axes[1].axvline(k, color='grey', lw=0.6, linestyle=':', alpha=0.7)
axes[1].set(xlabel='Age', ylabel='f₂(age)', title='GAM Component: Age (cubic spline)')

bar_colors = [C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE]
axes[2].bar(range(5), edu_coefs, color=bar_colors, alpha=0.75)
axes[2].axhline(0, color='grey', lw=0.7)
axes[2].set_xticks(range(5))
axes[2].set_xticklabels(edu_levels)
axes[2].set(xlabel='Education', ylabel='f₃(education)',
             title='GAM Component: Education')

plt.suptitle('GAM Components: wage ~ f₁(year) + f₂(age) + f₃(education)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Logistic GAM — Classification

In [ ]:
y_bin_all = (wage_arr > 250).astype(int)
print(f'High earners: {y_bin_all.sum()} ({y_bin_all.mean()*100:.1f}%)')

m_gam_log = LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs')
m_gam_log.fit(X_gam, y_bin_all)
print(f'Logistic GAM accuracy: {m_gam_log.score(X_gam, y_bin_all):.4f}')
print(f'Baseline (always predict 0): {1 - y_bin_all.mean():.4f}')

In [ ]:
# Plot age component on log-odds scale
f2_log = X_age_rng @ m_gam_log.coef_[0][n_year : n_year + n_age]
f2_log -= f2_log.mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(age_rng, f2_log, color=C_GREEN, lw=2.5, label='f₂(age) — log-odds contribution')
ax.axhline(0, color='grey', lw=0.7, linestyle='--')
for k in knots_age:
    ax.axvline(k, color='grey', lw=0.6, linestyle=':', alpha=0.7)
ax.set(xlabel='Age', ylabel='Log-odds contribution',
       title='Logistic GAM: Age Component for Pr(Wage > 250)')
ax.legend()
plt.tight_layout()
plt.show()

### Using pygam (optional — full GAM library)

In [ ]:
try:
    from pygam import LinearGAM, s, f
    
    # Encode education as integer
    edu_map = {'<HS': 0, 'HS': 1, '<Coll': 2, 'Coll': 3, '>Coll': 4}
    edu_int = np.array([edu_map[e] for e in edu_arr])
    X_pygam = np.column_stack([age_arr, year_arr, edu_int])
    
    gam = LinearGAM(s(0) + s(1) + f(2)).fit(X_pygam, wage_arr)
    print(f'pygam R² = {gam.statistics_["pseudo_r2"]["explained_deviance"]:.4f}')
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    titles = ['f(age)', 'f(year)', 'f(education)']
    for i, (ax, title) in enumerate(zip(axes, titles)):
        XX = gam.generate_X_grid(term=i)
        ax.plot(XX[:, i], gam.partial_dependence(term=i, X=XX), color=C_RED, lw=2.5)
        ax.plot(XX[:, i], gam.partial_dependence(term=i, X=XX, width=0.95)[1], 
                color=C_RED, lw=1.0, linestyle='--', alpha=0.5)
        ax.set(xlabel=titles[i].split('(')[1][:-1], ylabel=title, title=f'pygam: {title}')
    
    plt.suptitle('pygam: LinearGAM Component Plots', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('pygam not installed. Run: pip install pygam')
    print('Skipping pygam demo — manual GAM above covers the same concepts.')

---
## All methods side by side

A direct comparison of all six methods on the same wage ~ age data.

In [ ]:
x_rng = np.linspace(age_arr.min(), age_arr.max(), 300)

# 1. Degree-4 polynomial
pipe4 = Pipeline([('poly', PolynomialFeatures(4, include_bias=False)), ('lr', LinearRegression())])
pipe4.fit(age_arr.reshape(-1,1), wage_arr)
y_poly = pipe4.predict(x_rng.reshape(-1,1))

# 2. Step function (4 bins)
cuts_all = [18, 33.5, 49, 64.5, 80]
labs_all  = ['18–33', '33–49', '49–65', '65–80']
ac = pd.cut(age_arr, bins=cuts_all, labels=labs_all, include_lowest=True)
D  = pd.get_dummies(ac).astype(float)
m_s = LinearRegression().fit(D.values, wage_arr)
ac_r = pd.cut(x_rng, bins=cuts_all, labels=labs_all, include_lowest=True)
D_r  = pd.get_dummies(ac_r).astype(float)
y_step_all = m_s.predict(D_r.values)

# 3. Cubic spline
knots3 = np.quantile(age_arr, [0.25, 0.5, 0.75])
m_cs = LinearRegression().fit(truncated_power_basis(age_arr, knots3), wage_arr)
y_cs = m_cs.predict(truncated_power_basis(x_rng, knots3))

# 4. Smoothing spline
spl4 = UnivariateSpline(np.sort(age_arr), wage_arr[np.argsort(age_arr)], k=3, s=len(age_arr)*150)
y_ss = spl4(x_rng)

# 5. LOESS
lo5 = lowess(wage_arr, age_arr, frac=0.4, return_sorted=True)

# Plot
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
methods = [
    ('Degree-4 Polynomial',  x_rng, y_poly,     C_ORANGE, 'solid'),
    ('Step Function',        x_rng, y_step_all,  C_GREEN,  'solid'),
    ('Cubic Spline',         x_rng, y_cs,        C_RED,    'solid'),
    ('Smoothing Spline',     x_rng, y_ss,        C_BLUE,   'solid'),
    ('LOESS (span=0.4)',     lo5[:,0], lo5[:,1], C_PURPLE, 'solid'),
]
axes_flat = axes.ravel()
for ax, (name, x_, y_, color, ls) in zip(axes_flat, methods):
    ax.scatter(age_arr, wage_arr, alpha=0.12, s=12, color='steelblue')
    ax.plot(x_, y_, color=color, lw=2.5, linestyle=ls, label=name)
    ax.set(xlabel='Age', ylabel='Wage', title=name)
    ax.legend(fontsize=9)

# Last panel: all overlaid
ax = axes_flat[5]
ax.scatter(age_arr, wage_arr, alpha=0.12, s=12, color='steelblue')
ax.plot(x_rng, y_poly, color=C_ORANGE, lw=2, label='Polynomial')
ax.plot(x_rng, y_cs,   color=C_RED,    lw=2, label='Cubic Spline')
ax.plot(x_rng, y_ss,   color=C_BLUE,   lw=2, label='Smooth Spline')
ax.plot(lo5[:,0], lo5[:,1], color=C_PURPLE, lw=2, label='LOESS')
ax.set(xlabel='Age', ylabel='Wage', title='All Methods Overlaid')
ax.legend(fontsize=8)

plt.suptitle('Comparison: All Nonlinear Methods', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Exercises

Work through each exercise. A solution cell follows each problem — try it yourself first!

### Exercise 1 — Polynomial degree selection on the Auto dataset

The ISLR `Auto` dataset contains `mpg` (fuel efficiency) and `horsepower`.  
Their relationship is nonlinear.

**Task:**
1. Generate a synthetic Auto dataset: horsepower ∈ [50, 250], mpg = 60/(1 + 0.018·hp) + noise
2. Fit polynomial regression for degrees 1–8
3. Use 10-fold CV to compute the test MSE for each degree
4. Plot training MSE and CV MSE vs degree, and identify the best degree
5. Plot the best-fit curve on the data

In [ ]:
# Your code here


In [ ]:
# Solution — Exercise 1
rng_ex = np.random.default_rng(0)
n_auto = 392
hp = rng_ex.uniform(50, 250, n_auto)
mpg = 60 / (1 + 0.018*hp) + rng_ex.normal(0, 2.5, n_auto)

kf10 = KFold(n_splits=10, shuffle=True, random_state=0)
degrees = range(1, 9)
train_mse, cv_mse_ex = [], []

for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(d, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    pipe.fit(hp.reshape(-1,1), mpg)
    train_mse.append(mean_squared_error(mpg, pipe.predict(hp.reshape(-1,1))))
    scores = cross_val_score(pipe, hp.reshape(-1,1), mpg,
                             cv=kf10, scoring='neg_mean_squared_error')
    cv_mse_ex.append(-scores.mean())

best = list(degrees)[np.argmin(cv_mse_ex)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(list(degrees), train_mse, 'o-', color=C_BLUE, label='Training MSE')
axes[0].plot(list(degrees), cv_mse_ex, 's-', color=C_RED, label='10-fold CV MSE')
axes[0].axvline(best, color='grey', lw=1.2, linestyle='--', label=f'Best degree={best}')
axes[0].set(xlabel='Degree', ylabel='MSE', title='Auto: MSE vs Polynomial Degree')
axes[0].legend()

hp_rng = np.linspace(hp.min(), hp.max(), 300)
pipe_best = Pipeline([
    ('poly', PolynomialFeatures(best, include_bias=False)),
    ('lr',   LinearRegression())
])
pipe_best.fit(hp.reshape(-1,1), mpg)
axes[1].scatter(hp, mpg, alpha=0.35, s=20, color='steelblue')
axes[1].plot(hp_rng, pipe_best.predict(hp_rng.reshape(-1,1)),
             color=C_RED, lw=2.5, label=f'Degree-{best} polynomial')
axes[1].set(xlabel='Horsepower', ylabel='MPG',
            title=f'Best Polynomial Fit (degree={best})')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Best degree = {best}, CV MSE = {cv_mse_ex[best-1]:.3f}')

### Exercise 2 — Cubic spline vs polynomial: wage ~ age

**Task:**
1. Use the Wage dataset (already loaded)
2. Fit a **cubic spline** with 3 knots placed at the 25th, 50th, 75th percentiles of age
3. Fit a **degree-3 polynomial** with the same effective df (= 7)
4. Plot both fits overlaid on the scatter
5. Compare their 10-fold CV MSE — which is better?

In [ ]:
# Your code here


In [ ]:
# Solution — Exercise 2
age_v  = df['age'].values
wage_v = df['wage'].values

knots_ex2 = np.quantile(age_v, [0.25, 0.50, 0.75])

# Cubic spline (K=3 knots → 3+4=7 df)
X_cs_ex2 = truncated_power_basis(age_v, knots_ex2)
X_cs_rng = truncated_power_basis(x_rng, knots_ex2)
m_cs_ex2 = LinearRegression(fit_intercept=True).fit(X_cs_ex2, wage_v)
y_cs_ex2 = m_cs_ex2.predict(X_cs_rng)

# Degree-3 polynomial (1+3=4 params → match with intercept+3 poly terms)
# To match 7 df: use degree=6 polynomial (intercept + 6 terms = 7 params)
pipe_ex2 = Pipeline([
    ('poly', PolynomialFeatures(degree=6, include_bias=False)),
    ('lr',   LinearRegression())
])
pipe_ex2.fit(age_v.reshape(-1,1), wage_v)
y_poly_ex2 = pipe_ex2.predict(x_rng.reshape(-1,1))

# 10-fold CV
kf_ex2 = KFold(n_splits=10, shuffle=True, random_state=1)

def cv_mse_manual(X, y, kf):
    mses = []
    for tr, te in kf.split(X):
        m = LinearRegression(fit_intercept=True).fit(X[tr], y[tr])
        mses.append(mean_squared_error(y[te], m.predict(X[te])))
    return np.mean(mses)

cv_cs  = cv_mse_manual(np.column_stack([np.ones(len(age_v)), X_cs_ex2]), wage_v, kf_ex2)
cv_pol = -cross_val_score(pipe_ex2, age_v.reshape(-1,1), wage_v,
                          cv=kf_ex2, scoring='neg_mean_squared_error').mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(age_v, wage_v, alpha=0.15, s=15, color='steelblue')
ax.plot(x_rng, y_cs_ex2,   color=C_RED,    lw=2.5, label=f'Cubic Spline (7 df), CV MSE={cv_cs:.1f}')
ax.plot(x_rng, y_poly_ex2, color=C_BLUE,   lw=2.0, linestyle='--',
        label=f'Degree-6 Polynomial (7 df), CV MSE={cv_pol:.1f}')
for k in knots_ex2:
    ax.axvline(k, color='grey', lw=0.7, linestyle=':', alpha=0.6)
ax.set(xlabel='Age', ylabel='Wage', title='Cubic Spline vs Polynomial (same df=7)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f'Cubic spline CV MSE: {cv_cs:.2f}')
print(f'Polynomial    CV MSE: {cv_pol:.2f}')

### Exercise 3 — Smoothing spline with lambda selection

**Task:**
1. Use the Wage dataset (wage ~ age)
2. Fit smoothing splines for a grid of s values: `np.logspace(1, 6, 50)`
3. For each s, compute the number of effective knots (proxy for df)
4. Plot the fits for 3 different s values on the data
5. Use GCV (generalised cross-validation) to select the optimal s and plot the result

In [ ]:
# Your code here


In [ ]:
# Solution — Exercise 3
age_s3  = np.sort(df['age'].values)
wage_s3 = df['wage'].values[np.argsort(df['age'].values)]
x_rng3  = np.linspace(age_s3.min(), age_s3.max(), 300)
s_grid3 = np.logspace(1, 6, 50)

gcv_scores3 = []
n3 = len(age_s3)

for s_val in s_grid3:
    spl = UnivariateSpline(age_s3, wage_s3, k=3, s=s_val)
    residuals = wage_s3 - spl(age_s3)
    df_eff = max(len(spl.get_knots()), 2)
    gcv = np.mean(residuals**2) / (1 - df_eff/n3)**2
    gcv_scores3.append(gcv)

best_s3 = s_grid3[np.argmin(gcv_scores3)]
spl_best3 = UnivariateSpline(age_s3, wage_s3, k=3, s=best_s3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: multiple s values
axes[0].scatter(df['age'], df['wage'], alpha=0.15, s=15, color='steelblue')
for s_val, color, label in [
    (n3*5,    C_RED,    's=small (wiggly)'),
    (n3*200,  C_ORANGE, 's=medium'),
    (n3*3000, C_GREEN,  's=large (smooth)'),
]:
    spl = UnivariateSpline(age_s3, wage_s3, k=3, s=s_val)
    axes[0].plot(x_rng3, spl(x_rng3), color=color, lw=2.5, label=label)
axes[0].set(xlabel='Age', ylabel='Wage', title='Smoothing Splines: Different s Values')
axes[0].legend()

# Panel 2: GCV + best fit
ax2t = axes[1].twinx()
axes[1].semilogx(s_grid3, gcv_scores3, 'o-', color=C_BLUE, markersize=3)
axes[1].axvline(best_s3, color=C_RED, lw=1.5, linestyle='--')
axes[1].set(xlabel='s parameter (log scale)', ylabel='GCV Score',
            title=f'GCV optimal s = {best_s3:.0f}')
axes[1].tick_params(axis='y', labelcolor=C_BLUE)

plt.tight_layout()
plt.show()

# Plot best fit
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['age'], df['wage'], alpha=0.18, s=18, color='grey')
ax.plot(x_rng3, spl_best3(x_rng3), color=C_RED, lw=2.5,
        label=f'GCV-optimal smoothing spline (s={best_s3:.0f}, {len(spl_best3.get_knots())} knots)')
ax.set(xlabel='Age', ylabel='Wage', title='GCV-Optimal Smoothing Spline')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 4 — GAM with multiple predictors

**Task:**
1. Use the Wage dataset with all three predictors: age, year, education
2. Fit a GAM using:
   - Cubic spline with 3 knots for `age`
   - Linear term for `year`
   - Dummy variables for `education`
3. Compute the train R² and 10-fold CV R²
4. Compare to a baseline linear model (linear age + linear year + education dummies)
5. Plot the component plot for age from the GAM
6. Interpret: which predictor has the strongest nonlinear effect?

In [ ]:
# Your code here


In [ ]:
# Solution — Exercise 4
age_e4   = df['age'].values
year_e4  = df['year'].values
edu_e4   = df['education'].values
wage_e4  = df['wage'].values

# GAM features
knots_e4   = np.quantile(age_e4, [0.25, 0.50, 0.75])
X_age_e4   = truncated_power_basis(age_e4, knots_e4)
X_year_e4  = (year_e4 - year_e4.mean()).reshape(-1, 1)
X_edu_e4   = pd.get_dummies(edu_e4, drop_first=True).values.astype(float)
X_gam_e4   = np.column_stack([X_year_e4, X_age_e4, X_edu_e4])

m_gam_e4   = LinearRegression(fit_intercept=True).fit(X_gam_e4, wage_e4)
r2_train   = m_gam_e4.score(X_gam_e4, wage_e4)

# Baseline linear model
X_lin_e4  = np.column_stack([
    (year_e4 - year_e4.mean()),
    age_e4,
    X_edu_e4
])
m_lin_e4  = LinearRegression(fit_intercept=True).fit(X_lin_e4, wage_e4)
r2_lin    = m_lin_e4.score(X_lin_e4, wage_e4)

# 10-fold CV R²
kf_e4 = KFold(n_splits=10, shuffle=True, random_state=2)

def cv_r2(X, y, kf):
    r2s = []
    for tr, te in kf.split(X):
        m = LinearRegression(fit_intercept=True).fit(X[tr], y[tr])
        r2s.append(m.score(X[te], y[te]))
    return np.mean(r2s)

cv_r2_gam = cv_r2(X_gam_e4, wage_e4, kf_e4)
cv_r2_lin = cv_r2(X_lin_e4, wage_e4, kf_e4)

print('=' * 45)
print(f'{"Model":<20} {"Train R²":>10} {"CV R²":>10}')
print('-' * 45)
print(f'{"GAM (spline age)":<20} {r2_train:>10.4f} {cv_r2_gam:>10.4f}')
print(f'{"Linear baseline":<20} {r2_lin:>10.4f} {cv_r2_lin:>10.4f}')
print('=' * 45)

# Plot age component
n_yr_e4 = 1
n_ag_e4 = X_age_e4.shape[1]
age_rng_e4 = np.linspace(age_e4.min(), age_e4.max(), 200)
X_age_rng_e4 = truncated_power_basis(age_rng_e4, knots_e4)
f2_e4 = X_age_rng_e4 @ m_gam_e4.coef_[n_yr_e4 : n_yr_e4 + n_ag_e4]
f2_e4 -= f2_e4.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(age_rng_e4, f2_e4, color=C_RED, lw=2.5)
axes[0].axhline(0, color='grey', lw=0.7, linestyle='--')
for k in knots_e4:
    axes[0].axvline(k, color='grey', lw=0.6, linestyle=':', alpha=0.7)
axes[0].set(xlabel='Age', ylabel='f(age) — centred',
             title='GAM Component: Age')

# Education bar chart
edu_coefs_e4 = np.concatenate([[0], m_gam_e4.coef_[n_yr_e4+n_ag_e4:]])
edu_coefs_e4 -= edu_coefs_e4.mean()
edu_labels = ['<HS', 'HS', '<Coll', 'Coll', '>Coll']
axes[1].bar(range(5), edu_coefs_e4,
            color=[C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE], alpha=0.75)
axes[1].axhline(0, color='grey', lw=0.7)
axes[1].set_xticks(range(5))
axes[1].set_xticklabels(edu_labels)
axes[1].set(xlabel='Education', ylabel='f(education) — centred',
             title='GAM Component: Education')

plt.suptitle('GAM Components — Exercise 4', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('- Education has the strongest effect: >Coll earns ~35 more than <HS on average.')
print('- Age has a nonlinear hump: earnings peak around 45-55, then decline.')
print('- Year is nearly linear with a small positive slope.')
print(f'- GAM improves over linear: CV R² {cv_r2_gam:.4f} vs {cv_r2_lin:.4f}')

---
## Summary

| Method | Key idea | Python | Pros | Cons |
|--------|----------|--------|------|------|
| **Polynomial** | X, X², …, Xᵈ | `PolynomialFeatures` | Simple, interpretable | Tail instability |
| **Step function** | bins → dummies | `pd.cut + get_dummies` | Intuitive, interaction-friendly | Misses smooth trends |
| **Cubic spline** | Truncated power basis | Manual / `patsy.cr` | Smooth, flexible | Need to choose knots |
| **Natural spline** | Linear tails | `scipy.UnivariateSpline` | Stable at boundaries | Slightly more complex |
| **Smoothing spline** | Roughness penalty | `UnivariateSpline(s=...)` | Automatic knot selection | Single λ tuning |
| **LOESS** | Local weighted OLS | `statsmodels.lowess` | Very flexible | Slow for large n, no global formula |
| **GAM** | Sum of f_j(X_j) | Manual / `pygam` | Interpretable components | Additive (misses interactions) |

**Key takeaway**: All methods fit inside the linear model framework — transform predictors, then run OLS or logistic regression as usual.